# MDD Challenge 2025 — Training Notebook (Kaggle P100)

**GPU**: P100 (16GB)  
**Run mode**: Set `RUN_STAGE` to control which stage executes.

```
RUN_STAGE = 'stage1'   → Minimal baseline (wav2vec2-vn, no preprocessing)
RUN_STAGE = 'stage2'   → Preprocessing ablations (A–F, one at a time)
RUN_STAGE = 'stage3'   → Model comparison (hubert / wav2vec2-100h)
```

**Score formula**: `Score = 0.5×F1 + 0.4×(1−DER) + 0.1×(1−PER)`  
**Trivial baseline (B0)**: Score = 0.4994 (predict canonical = no errors)

**Fixed split** (never change):  
- Train: 2720 samples, 22 speakers (incl. TUYEN, ADULT)  
- Valid: 460 samples, speakers S0008 (6.2% error) + S0003 (15.4% error)

In [ ]:
# ── Pull code from GitHub ────────────────────────────────────────────────────
import subprocess, os, sys
from pathlib import Path

REPO_URL = 'https://github.com/quangzp/mdd-challenge.git'
BRANCH   = 'feat/giang'
REPO_DIR = Path('/kaggle/working/mdd-challenge')

if not REPO_DIR.exists():
    subprocess.check_call([
        'git', 'clone', '--branch', BRANCH, '--depth', '1',
        REPO_URL, str(REPO_DIR)
    ])
    print(f'Cloned → {REPO_DIR}')
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
    print(f'Updated existing clone')

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f'cwd: {os.getcwd()}')
print('Files:', [p.name for p in sorted(REPO_DIR.iterdir()) if not p.name.startswith('.')])


## 1. Setup

In [ ]:
# ── Environment check — no installation needed ──────────────────────────────
# Kaggle already has transformers 5.x + huggingface-hub 1.x + accelerate preinstalled.
# Previous attempts to install transformers 4.x caused conflicts with hf-hub>=1.0.
import transformers, accelerate, torch
print(f'torch         : {torch.__version__}')
print(f'transformers  : {transformers.__version__}')
print(f'accelerate    : {accelerate.__version__}')
if torch.cuda.is_available():
    cap  = torch.cuda.get_device_capability()
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU           : {name}  sm_{cap[0]}{cap[1]}  {vram:.1f} GB')
    # Quick kernel smoke-test
    try:
        _ = torch.zeros(2, 2, device='cuda').matmul(torch.ones(2, 2, device='cuda'))
        print('CUDA kernel   : OK')
    except Exception as e:
        print(f'CUDA kernel FAILED: {e}')
        print('→ Switch to T4 GPU (Accelerator → GPU T4 x1) and restart kernel')
        raise
else:
    print('No GPU — will train on CPU (slow)')
print('Environment OK')


In [ ]:
# ── Stage selector ───────────────────────────────────────────────────────────
RUN_STAGE = 'stage6'   # 'stage1'|'stage2'|'stage3'|'stage4_diag'|'stage5'|'stage6'

# Stage 2 ablation selector (only used when RUN_STAGE='stage2')
# Run one at a time; each is independent from Stage 1 baseline
ABLATION = 'C'  # 'A'=normalize_amp | 'B'=trim_silence | 'C'=gain_aug
               # 'D'=noise_aug | 'E'=spec_augment | 'F'=no_adult

# Stage 3 model selector (only used when RUN_STAGE='stage3')
STAGE3_MODEL = 'hubert'  # 'hubert' | 'wav2vec2-100h'

# ── Global hyperparams (identical across all experiments) ───────────────────
EPOCHS       = 10
BATCH_SIZE   = 4
GRAD_ACCUM   = 4         # effective batch = 16
ENCODER_LR   = 2e-5
HEAD_LR      = 2e-3
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_SECONDS  = 15        # truncate audio at 15s
SEED         = 42
BLANK_PENALTIES = [0.0, 0.5, 1.0, 1.5, 2.0]  # sweep at decode time

# Stage 6 eval protocol constants
FIXED_PENALTY    = 0.0   # fixed; not tuned per-run (consistent with per-epoch callback)
STAGE6_BEST_CKPT = '/kaggle/working/outputs/stage6_best_ckpt'
EPOCH_LOG_S6     = '/kaggle/working/results/stage6_epoch_metrics.json'

print(f'Stage: {RUN_STAGE}  |  Ablation: {ABLATION}  |  Model: {STAGE3_MODEL}')

In [ ]:
import os, re, csv, json, math, wave, random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from dataclasses import dataclass

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoFeatureExtractor, Wav2Vec2ForCTC, HubertForCTC,
    TrainingArguments, Trainer,
    get_linear_schedule_with_warmup, set_seed,
)

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
# ── GPU compatibility check ──────────────────────────────────────────────────
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    name = torch.cuda.get_device_name(0)
    print(f'GPU: {name}  (sm_{cap[0]}{cap[1]})')
    # Verify CUDA kernels work on this device
    try:
        _t = torch.zeros(4, 4, device='cuda').sum()
        print(f'CUDA kernel test: OK (result={float(_t)})')
    except Exception as e:
        print(f'CUDA kernel test FAILED: {e}')
        print('→ Switch to T4 GPU in Kaggle settings (Accelerator → GPU T4 x1)')
        raise RuntimeError('GPU not usable — switch to T4') from e
    # Recommend T4 if sm < 70 and torch >= 2.5
    if cap[0] < 7:
        tv = tuple(int(x) for x in torch.__version__.split('.')[:2])
        if tv >= (2, 5):
            print(f'WARNING: torch {torch.__version__} may not support sm_{cap[0]}{cap[1]}')
            print('Recommendation: switch to T4 GPU (sm_75) or pin torch<2.5')


## 2. Paths & Data Loading

In [ ]:
# ── Resolve paths (local or Kaggle) ─────────────────────────────────────────
def _find(candidates):
    for p in candidates:
        if Path(p).exists(): return Path(p)
    raise FileNotFoundError(f'None found: {candidates}')

DATA_ROOT = _find([
    # Kaggle: dataset honggiangtrnh/mdd-challenge-2025
    '/kaggle/input/mdd-challenge-2025',
    # Local fallback
    'MDD-Challenge-2025-training-set',
])
AUDIO_DIR = DATA_ROOT / 'audio_data' / 'train'
META_DIR  = DATA_ROOT / 'metadata'

SPLITS_DIR = Path('/kaggle/working/splits')
SPLITS_DIR.mkdir(exist_ok=True)
Path('/kaggle/working/results').mkdir(exist_ok=True)

# Override results/splits dirs to write to /kaggle/working (writable on Kaggle)
import os
SPLITS_DIR_STR  = str(SPLITS_DIR)
RESULTS_DIR_STR = '/kaggle/working/results'

print(f'DATA_ROOT : {DATA_ROOT}')
print(f'AUDIO_DIR : {AUDIO_DIR}  exists={AUDIO_DIR.exists()}')
print(f'META_DIR  : {META_DIR}  exists={META_DIR.exists()}')


In [ ]:
# ── Utilities (inline copy of utils.py for Kaggle self-containedness) ────────
VALID_SPEAKERS = frozenset(['S0008', 'S0003'])
BLANK, UNK = '<blank>', '<unk>'

def get_speaker_id(path):
    stem = Path(path).stem
    m = re.search(r'(S\d+)', stem)
    if m: return m.group(1)
    return 'TUYEN' if 'tuyen' in stem.lower() else 'ADULT'

def norm_phones(s):
    return ' '.join(str(s).replace('*','').replace('$','').split())

def tokenize(s):
    return norm_phones(s).split()

def load_wav_f32(path):
    with wave.open(str(path), 'rb') as wf:
        sr, sw, nch = wf.getframerate(), wf.getsampwidth(), wf.getnchannels()
        frames = wf.readframes(wf.getnframes())
    dtype = {1: np.uint8, 2: np.int16, 4: np.int32}[sw]
    scale = {1: 128.0,   2: 32768.0,  4: 2147483648.0}[sw]
    y = np.frombuffer(frames, dtype=dtype).astype(np.float32)
    y = (y - 128.0)/128.0 if sw==1 else y/scale
    if nch > 1: y = y.reshape(-1, nch).mean(axis=1)
    return y.copy(), sr

def evaluate_on_valid(gt_c, gt_t, preds, tag=''):
    import sys; sys.path.insert(0, str(Path('.').resolve()))
    import evaluate as ev
    gt_p, res_p = '/tmp/_gt.csv', '/tmp/_res.csv'
    with open(gt_p, 'w', newline='') as f:
        w = csv.DictWriter(f, ['canonical','transcript']); w.writeheader()
        for c,t in zip(gt_c, gt_t): w.writerow({'canonical':c,'transcript':t})
    with open(res_p, 'w', newline='') as f:
        w = csv.DictWriter(f, ['predict']); w.writeheader()
        for p in preds: w.writerow({'predict':p})
    f1  = ev.compute_f1( gt_p, res_p)
    per = ev.compute_per(gt_p, res_p)
    der = ev.compute_der(gt_p, res_p)
    sc  = 0.5*f1 + 0.4*(1-der) + 0.1*(1-per)
    if tag: print(f'{tag:45s}  F1={f1:.4f}  PER={per:.4f}  DER={der:.4f}  Score={sc:.4f}')
    return {'f1':f1,'per':per,'der':der,'score':sc}

def greedy_ctc(logits, id2phone, phone2id, penalty=0.0):
    adj = logits.copy()
    adj[..., phone2id[BLANK]] -= penalty
    ids = np.argmax(adj, axis=-1)
    res = []
    for seq in ids:
        out, prev = [], None
        for i in seq:
            if i==prev: continue
            prev=int(i)
            if prev==phone2id[BLANK]: continue
            out.append(id2phone[prev] if prev<len(id2phone) else UNK)
        res.append(' '.join(out))
    return res

def save_results(d, path='/kaggle/working/results/results_log.json'):
    existing = json.load(open(path)) if Path(path).exists() else {}
    existing.update(d)
    with open(path,'w') as f: json.dump(existing, f, indent=2, ensure_ascii=False)
    print(f'Saved → {path}')

print('Utilities loaded.')

In [ ]:
# ── Stage 0: build split + vocab (runs in <10s) ──────────────────────────────
import json as _json
VOCAB_PATH = Path(SPLITS_DIR_STR) / 'phone_vocab.json'

if not VOCAB_PATH.exists():
    print('Running Stage 0 inline...')
    df_t = pd.read_csv(META_DIR / 'train.csv')
    df_p = pd.read_csv(META_DIR / 'train_phones.csv')
    df_t['speaker_id'] = df_t['path'].map(get_speaker_id)
    df_p['speaker_id'] = df_t['speaker_id'].values
    df_p['c_norm']     = df_p['canonical'].map(norm_phones)
    df_p['t_norm']     = df_p['transcript'].map(norm_phones)
    df_p['ph_error']   = (df_p['c_norm'] != df_p['t_norm'])
    vm = df_t['speaker_id'].isin(VALID_SPEAKERS)
    train_df = df_t[~vm].copy().reset_index(drop=True)
    valid_df = df_t[ vm].copy().reset_index(drop=True)
    train_ph = df_p[~vm].copy().reset_index(drop=True)
    valid_ph = df_p[ vm].copy().reset_index(drop=True)
    counter = Counter()
    for s in train_ph['t_norm']: counter.update(tokenize(s))
    for s in df_p['c_norm']:     counter.update(tokenize(s))
    id2phone = [BLANK, UNK] + sorted(counter.keys())
    phone2id = {p: i for i, p in enumerate(id2phone)}
    sp = Path(SPLITS_DIR_STR)
    train_df.to_csv(sp/'train_ids.csv',    index=False)
    valid_df.to_csv(sp/'valid_ids.csv',    index=False)
    train_ph.to_csv(sp/'train_phones.csv', index=False)
    valid_ph.to_csv(sp/'valid_phones.csv', index=False)
    with open(sp/'phone_vocab.json','w') as f:
        _json.dump({'id2phone':id2phone,'phone2id':phone2id}, f, ensure_ascii=False)
    print(f'Stage 0 done. Vocab={len(id2phone)}, Train={len(train_df)}, Valid={len(valid_df)}')
else:
    print('Stage 0 artifacts found, loading...')
    sp = Path(SPLITS_DIR_STR)
    train_df = pd.read_csv(sp/'train_ids.csv')
    valid_df = pd.read_csv(sp/'valid_ids.csv')
    train_ph = pd.read_csv(sp/'train_phones.csv')
    valid_ph = pd.read_csv(sp/'valid_phones.csv')
    with open(sp/'phone_vocab.json') as f:
        vd = _json.load(f)
    id2phone, phone2id = vd['id2phone'], vd['phone2id']
    print(f'Loaded. Vocab={len(id2phone)}, Train={len(train_df)}, Valid={len(valid_df)}')

VALID_C = valid_ph['c_norm'].tolist()
VALID_T = valid_ph['t_norm'].tolist()
assert set(train_df['speaker_id']).isdisjoint(VALID_SPEAKERS), 'Speaker leak!'
assert len(id2phone) == 123, f'Vocab size mismatch: {len(id2phone)}'
print(f'Split OK — Train:{len(train_df)}, Valid:{len(valid_df)}, Vocab:{len(id2phone)}')


## 3. Preprocessing Functions

In [ ]:
MAX_SAMPLES = MAX_SECONDS * 16000

# ── Preprocessing ops ────────────────────────────────────────────────────────
def normalize_amp(y, target_peak=0.9):
    peak = np.abs(y).max()
    return y if peak < 1e-6 else y * (target_peak / peak)

def trim_silence(y, sr=16000, threshold_db=-45,
                 frame_ms=25, hop_ms=10, min_dur_sec=0.3):
    fl = int(sr*frame_ms/1000); hl = int(sr*hop_ms/1000)
    e  = [np.mean(y[i:i+fl]**2) for i in range(0, max(1,len(y)-fl), hl)]
    db = 10*np.log10(np.array(e)+1e-10)
    act = db > threshold_db
    if not act.any(): return y
    s = max(0, int(np.argmax(act))*hl - fl)
    e = min(len(y), (len(act)-int(np.argmax(act[::-1])))*hl+fl)
    trimmed = y[s:e]
    return y if len(trimmed)/sr < min_dur_sec else trimmed

def aug_gain(y, lo=-6., hi=6.):
    return np.clip(y * 10**(np.random.uniform(lo, hi)/20.), -1., 1.)

def aug_noise(y, snr_lo=20., snr_hi=35., prob=0.3):
    if np.random.rand() > prob: return y
    sp  = np.mean(y**2) + 1e-10
    np_ = sp / 10**(np.random.uniform(snr_lo, snr_hi)/10.)
    return np.clip(y + np.random.normal(0., np_**0.5, y.shape).astype(np.float32), -1., 1.)

# ── Build audio pipeline based on stage/ablation ────────────────────────────
def build_audio_pipeline(stage, ablation=None, no_adult=False):
    """Returns (preprocess_fn, augment_fn, use_adult) for each experiment.
    All Stage 2 experiments are INDEPENDENT from Stage 1 — not stacked.
    """
    # Stage 1: minimal — only truncation
    def preproc_stage1(y): return y[:MAX_SAMPLES]
    def augment_noop(y):   return y

    if stage == 'stage1':
        return preproc_stage1, augment_noop, True  # include adult

    if stage == 'stage2':
        # Each ablation adds EXACTLY ONE thing over Stage 1 baseline
        if ablation == 'A':  # normalize_amp only
            def p(y): return normalize_amp(y[:MAX_SAMPLES])
            return p, augment_noop, True
        if ablation == 'B':  # trim_silence only
            def p(y): return trim_silence(y, 16000)[:MAX_SAMPLES]
            return p, augment_noop, True
        if ablation == 'C':  # gain augment only
            def p(y): return y[:MAX_SAMPLES]
            def a(y): return aug_gain(y)
            return p, a, True
        if ablation == 'D':  # noise augment only
            def p(y): return y[:MAX_SAMPLES]
            def a(y): return aug_noise(y)
            return p, a, True
        if ablation == 'E':  # spec_augment (handled in model config)
            return preproc_stage1, augment_noop, True
        if ablation == 'F':  # exclude adult data
            return preproc_stage1, augment_noop, False

    # Stage 3: use best_preproc from Stage 2 (update after Stage 2 done)
    # Default: Stage 1 config until Stage 2 results are in
    if stage == 'stage3':
        # best_preproc = B (trim_silence) + F (no_adult)
        def p(y): return trim_silence(y, 16000)[:MAX_SAMPLES]
        return p, augment_noop, False  # False = exclude adult

    raise ValueError(f'Unknown stage/ablation: {stage}/{ablation}')

print('Preprocessing functions defined.')

# Shared preprocessing for Stage 5/6 (trim_silence + truncate)
def p_trim(y): return trim_silence(y, 16000)[:MAX_SAMPLES]


## 4. Dataset & Collator

In [ ]:
class MDDDataset(Dataset):
    def __init__(self, df, ph_df, phone2id, preproc_fn, augment_fn,
                 is_train=True):
        self.df       = df.reset_index(drop=True)
        self.ph       = ph_df.reset_index(drop=True)
        self.p2i      = phone2id
        self.preproc  = preproc_fn
        self.augment  = augment_fn
        self.is_train = is_train

        # Precompute audio lengths from WAV headers (fast — no decoding)
        # Required by group_by_length=True (LengthGroupedSampler)
        self.lengths = []
        for _, row in self.df.iterrows():
            p = AUDIO_DIR / Path(row['path']).name
            try:
                with wave.open(str(p), 'rb') as wf:
                    self.lengths.append(wf.getnframes())
            except Exception:
                self.lengths.append(0)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ph  = self.ph.iloc[idx]
        y, sr = load_wav_f32(AUDIO_DIR / Path(row['path']).name)
        y = self.preproc(y)
        if self.is_train:
            y = self.augment(y)
        labels = [self.p2i.get(t, self.p2i[UNK])
                  for t in tokenize(ph['t_norm'])]
        return {'input_values': y.astype(np.float32),
                'labels': labels,
                'id': row['id'],
                'ph_error': bool(ph['ph_error'])}


@dataclass
class MDDCollator:
    fe: object
    sr: int = 16000
    pad_id: int = -100

    def __call__(self, batch):
        out = self.fe([b['input_values'] for b in batch],
                      sampling_rate=self.sr, padding=True,
                      return_attention_mask=True,  # critical for CTC
                      return_tensors='pt')
        labels = [torch.tensor(b['labels'], dtype=torch.long) for b in batch]
        maxL   = max(len(l) for l in labels)
        pad    = torch.full((len(labels), maxL), self.pad_id, dtype=torch.long)
        for i, l in enumerate(labels): pad[i, :len(l)] = l
        out['labels'] = pad
        return out

print('Dataset & Collator defined.')


## 5. Model Factory

In [ ]:
MODEL_CONFIGS = {
    'wav2vec2-vn':   'nguyenvulebinh/wav2vec2-base-vietnamese-250h',
    'hubert':        'facebook/hubert-base-ls960',
    'wav2vec2-100h': 'facebook/wav2vec2-base-100h',
}
MODEL_CLASSES = {
    'wav2vec2-vn':   Wav2Vec2ForCTC,
    'hubert':        HubertForCTC,
    'wav2vec2-100h': Wav2Vec2ForCTC,
}

def build_model(model_key, vocab_size, blank_id, enable_spec_augment=False):
    model_name  = MODEL_CONFIGS[model_key]
    model_class = MODEL_CLASSES[model_key]
    fe = AutoFeatureExtractor.from_pretrained(model_name)
    model = model_class.from_pretrained(
        model_name,
        vocab_size=vocab_size,
        pad_token_id=blank_id,
        ctc_loss_reduction='mean',
        ctc_zero_infinity=True,
        ignore_mismatched_sizes=True,
    )
    model.config.apply_spec_augment = enable_spec_augment
    if enable_spec_augment:
        model.config.mask_time_prob    = 0.05
        model.config.mask_feature_prob = 0.0
    else:
        model.config.mask_time_prob    = 0.0
        model.config.mask_feature_prob = 0.0
    model.freeze_feature_encoder()
    return model, fe

def build_optimizer_scheduler(model, n_train, epochs, batch, accum,
                               enc_lr, head_lr, wd, warmup_ratio):
    head_ids    = {id(p) for p in model.lm_head.parameters()}
    enc_params  = [p for p in model.parameters()
                   if id(p) not in head_ids and p.requires_grad]
    head_params = [p for p in model.lm_head.parameters() if p.requires_grad]
    steps  = math.ceil(n_train / (batch * accum)) * epochs
    warmup = int(steps * warmup_ratio)
    opt = torch.optim.AdamW(
        [{'params': enc_params, 'lr': enc_lr},
         {'params': head_params,'lr': head_lr}],
        weight_decay=wd)
    sch = get_linear_schedule_with_warmup(opt, warmup, steps)
    print(f'Optimizer: enc_lr={enc_lr}, head_lr={head_lr}')
    print(f'Steps={steps}, warmup={warmup}')
    print(f'Enc params: {sum(p.numel() for p in enc_params):,}')
    print(f'Head params: {sum(p.numel() for p in head_params):,}')
    return opt, sch

print('Model factory defined.')

## 6. Training & Evaluation Runner

In [ ]:
from transformers.trainer_pt_utils import LengthGroupedSampler
from torch.utils.data import RandomSampler, SequentialSampler

class MDDTrainer(Trainer):
    """Trainer subclass: fixes group_by_length for custom torch Dataset.
    Trainer 5.x only supports length-grouping for HF datasets.Dataset,
    not plain torch Dataset. We read dataset.lengths directly.
    """
    def _get_train_sampler(self, train_dataset=None):
        if train_dataset is None:
            train_dataset = self.train_dataset
        if not self.args.group_by_length or not hasattr(train_dataset, 'lengths'):
            return RandomSampler(train_dataset)
        return LengthGroupedSampler(
            self.args.train_batch_size * self.args.gradient_accumulation_steps,
            lengths=train_dataset.lengths,
            dataset=train_dataset,
        )

    def _get_eval_sampler(self, eval_dataset=None):
        if eval_dataset is None:
            eval_dataset = self.eval_dataset
        return SequentialSampler(eval_dataset)


def run_experiment(exp_name, model_key, train_df, train_ph, valid_df, valid_ph,
                   preproc_fn, augment_fn,
                   enable_spec_augment=False,
                   output_subdir=None,
                   epochs=None, epoch_callback=None):
    """Full train + eval loop. Returns best metrics dict."""
    n_epochs = epochs if epochs is not None else EPOCHS
    print(f'\n{"="*60}')
    print(f'Experiment: {exp_name}  |  Model: {model_key}')
    print(f'Train: {len(train_df)}  |  Valid: {len(valid_df)}')
    print(f'SpecAugment: {enable_spec_augment}')
    print(f'{"="*60}')

    out_dir = Path(f'outputs/{output_subdir or exp_name}')
    out_dir.mkdir(parents=True, exist_ok=True)

    # Build model
    model, fe = build_model(model_key, len(id2phone), phone2id[BLANK],
                            enable_spec_augment=enable_spec_augment)

    # Datasets
    train_ds = MDDDataset(train_df, train_ph, phone2id,
                          preproc_fn, augment_fn, is_train=True)
    valid_ds = MDDDataset(valid_df, valid_ph, phone2id,
                          preproc_fn, lambda y: y, is_train=False)
    collator = MDDCollator(fe=fe)

    # Sanity check first batch
    sample_batch = collator([train_ds[0], train_ds[1]])
    with torch.no_grad():
        init_loss = model(**{k: v for k,v in sample_batch.items()}).loss
    print(f'Initial CTC loss (sanity): {float(init_loss):.4f}')
    assert not torch.isnan(init_loss), 'NaN loss at init — check vocab/model'

    # Optimizer + scheduler
    opt, sch = build_optimizer_scheduler(
        model, len(train_df), n_epochs, BATCH_SIZE, GRAD_ACCUM,
        ENCODER_LR, HEAD_LR, WEIGHT_DECAY, WARMUP_RATIO
    )

    # Training args
    training_args = TrainingArguments(
        output_dir=str(out_dir / 'checkpoints'),
        num_train_epochs=n_epochs,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        # fp16 disabled: P100 is sm_60, some new accelerate kernels not compiled for it
        fp16=False,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        group_by_length=True,
        remove_unused_columns=False,
        report_to='none',
        logging_steps=30,
        seed=SEED,
    )

    trainer = MDDTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        data_collator=collator,
        optimizers=(opt, sch),
    )

    if epoch_callback is not None:
        epoch_callback.bind(model, fe, valid_df, valid_ph, preproc_fn)
        trainer.add_callback(epoch_callback)

    train_result = trainer.train()
    print(f'Training complete. Final train loss: {train_result.training_loss:.4f}')

    # Load best checkpoint if callback saved one
    if (epoch_callback is not None
            and getattr(epoch_callback, 'best_ckpt_dir', None)
            and Path(getattr(epoch_callback, 'best_ckpt_dir', '')).exists()):
        print(f'\nLoading best checkpoint from ep{epoch_callback._best_epoch}'
              f' (Score={epoch_callback._best_score:.4f})')
        model_class = MODEL_CLASSES[model_key]
        model = model_class.from_pretrained(epoch_callback.best_ckpt_dir).to(device)
        trainer.model = model  # update trainer for predict()

    # ── Decode & evaluate ────────────────────────────────────────────────────
    pred_out = trainer.predict(valid_ds)
    logits   = pred_out.predictions   # (n_valid, T, vocab)

    # Blank frame ratio (diagnostic)
    raw_ids  = np.argmax(logits, axis=-1)
    blank_ratio = (raw_ids == phone2id[BLANK]).mean()
    print(f'Blank frame ratio: {blank_ratio:.3f}')

    # Diagnostic penalty sweep (not used for primary result)
    print('\n── Penalty sweep (diagnostic) ──')
    for penalty in BLANK_PENALTIES:
        preds_tmp = greedy_ctc(logits, id2phone, phone2id, penalty=penalty)
        empty  = sum(1 for p in preds_tmp if not p.strip())
        lengths = [len(p.split()) for p in preds_tmp]
        evaluate_on_valid(VALID_C, VALID_T, preds_tmp,
                          f'  {exp_name} penalty={penalty:.1f} [diag]')
        print(f'    empty={empty}/{len(preds_tmp)}, '
              f'mean_len={np.mean(lengths):.1f}')

    # Primary result: FIXED_PENALTY (not tuned per-run)
    best_penalty  = FIXED_PENALTY
    preds_fixed   = greedy_ctc(logits, id2phone, phone2id, penalty=FIXED_PENALTY)
    best_metrics  = evaluate_on_valid(VALID_C, VALID_T, preds_fixed,
                                       f'{exp_name} FIXED_PENALTY={FIXED_PENALTY}')
    best_score    = best_metrics['score']
    best_preds    = preds_fixed
    print(f'\nPrimary: penalty={best_penalty}, Score={best_score:.4f}')

    # Per-speaker breakdown (bias check)
    # best_preds already set from FIXED_PENALTY above
    for spk in VALID_SPEAKERS:
        mask = valid_df['speaker_id'] == spk
        idx  = [i for i,m in enumerate(mask) if m]
        m = evaluate_on_valid(
            [VALID_C[i] for i in idx],
            [VALID_T[i] for i in idx],
            [best_preds[i] for i in idx],
            f'  {exp_name} [{spk}]'
        )

    # Save model + results
    trainer.save_model(str(out_dir / 'final_model'))
    fe.save_pretrained(str(out_dir / 'final_model'))

    result_record = {
        **best_metrics,
        'best_blank_penalty': best_penalty,
        'blank_frame_ratio': float(blank_ratio),
        'train_loss': float(train_result.training_loss),
        'model': model_key,
        'n_train': len(train_df),
        'n_valid': len(valid_df),
    }
    save_results({exp_name: result_record})
    return result_record

print('Runner defined.')

In [ ]:
# ── Per-epoch metrics callback (Stage 5) ────────────────────────────────────
from transformers import TrainerCallback

def _full_metrics_inline(gt_c, gt_t, preds):
    """Compute P/R/F1/PER/DER/Score from evaluate.py logic (P/R inline)."""
    import csv
    sys.path.insert(0, '/kaggle/working/mdd-challenge')
    import evaluate as ev
    from evaluate import _align_pair
    gp='/tmp/_em_gt.csv'; rp='/tmp/_em_res.csv'
    with open(gp,'w',newline='') as f:
        w=csv.DictWriter(f,['canonical','transcript']); w.writeheader()
        for c,t in zip(gt_c,gt_t): w.writerow({'canonical':c,'transcript':t})
    with open(rp,'w',newline='') as f:
        w=csv.DictWriter(f,['predict']); w.writeheader()
        for p in preds: w.writerow({'predict':p})
    gt_rows=list(csv.DictReader(open(gp))); res_rows=list(csv.DictReader(open(rp)))
    cor_nocor=sub_sub=sub_sub1=sub_nosub=0
    ins_ins=ins_ins1=ins_noins=del_del=del_del1=del_nodel=0
    for gr,rr in zip(gt_rows,res_rows):
        rs,hs,op_rh=_align_pair(gr['canonical'],gr['transcript'])
        hs2,os2,op_ho=_align_pair(gr['transcript'],rr['predict'])
        rs3,os3,op_ro=_align_pair(gr['canonical'],rr['predict'])
        flag=0
        for ii in range(len(rs)):
            if rs[ii]=='<eps>': continue
            while flag<len(rs3) and rs3[flag]=='<eps>': flag+=1
            if flag<len(rs3) and rs[ii]==rs3[flag]:
                if op_rh[ii]=='D' and op_ro[flag]=='D': del_del+=1
                elif op_rh[ii]=='D' and op_ro[flag]!='D' and op_ro[flag]!='C': del_del1+=1
                elif op_rh[ii]=='D' and op_ro[flag]=='C': del_nodel+=1
                flag+=1
        flag=0
        for ii in range(len(hs)):
            if hs[ii]=='<eps>': continue
            while flag<len(hs2) and hs2[flag]=='<eps>': flag+=1
            if flag<len(hs2) and hs[ii]==hs2[flag]:
                if op_rh[ii]=='C' and op_ho[flag]!='C': cor_nocor+=1
                if op_rh[ii]=='S' and op_ho[flag]=='C': sub_sub+=1
                elif op_rh[ii]=='S' and op_ho[flag]!='C' and rs[ii]!=os2[flag]: sub_sub1+=1
                elif op_rh[ii]=='S' and op_ho[flag]!='C' and rs[ii]==os2[flag]: sub_nosub+=1
                if op_rh[ii]=='I' and op_ho[flag]=='C': ins_ins+=1
                elif op_rh[ii]=='I' and op_ho[flag]!='C' and op_ho[flag]!='D': ins_ins1+=1
                elif op_rh[ii]=='I' and op_ho[flag]=='D': ins_noins+=1
                flag+=1
    TR=sub_sub+sub_sub1+del_del+del_del1+ins_ins+ins_ins1
    FR=cor_nocor; FA=sub_nosub+ins_noins+del_nodel
    prec=TR/(TR+FR) if(TR+FR)>0 else 0.0
    rec=TR/(TR+FA) if(TR+FA)>0 else 0.0
    f1=2*prec*rec/(prec+rec) if(prec+rec)>0 else 0.0
    per=ev.compute_per(gp,rp); der=ev.compute_der(gp,rp)
    return {'f1':f1,'prec':prec,'rec':rec,'per':per,'der':der,
            'score':0.5*f1+0.4*(1-der)+0.1*(1-per)}


class EpochMetricsCallback(TrainerCallback):
    """Logs P/R/F1/PER + subgroup PER (no-error vs error) after each eval epoch.
    Decisive test for Stage 5: does precision climb or plateau with epochs.
    """
    def __init__(self, valid_c, valid_t, id2phone, phone2id,
                 penalty=0.0, out_path=None, error_mask=None,
                 best_ckpt_dir=None):
        self.valid_c=valid_c; self.valid_t=valid_t
        self.id2phone=id2phone; self.phone2id=phone2id
        self.penalty=penalty; self.out_path=out_path
        self.error_mask=error_mask  # bool array over valid samples
        self.best_ckpt_dir=best_ckpt_dir
        self._best_score=-1.0; self._best_epoch=-1
        self.history=[]
        self._model=self._fe=self._vdf=self._vph=self._preproc=None

    def bind(self, model, fe, valid_df, valid_ph, preproc_fn):
        self._model=model; self._fe=fe
        self._vdf=valid_df; self._vph=valid_ph; self._preproc=preproc_fn

    def on_evaluate(self, args, state, control, **kwargs):
        model = kwargs.get('model', self._model)
        if model is None: return
        ds = MDDDataset(self._vdf, self._vph, self.phone2id,
                        self._preproc, lambda y:y, is_train=False)
        loader = torch.utils.data.DataLoader(ds, batch_size=8,
                     collate_fn=MDDCollator(fe=self._fe), shuffle=False)
        logits=[]
        model.eval()
        with torch.no_grad():
            for b in loader:
                b={k:v.to(device) if hasattr(v,'to') else v for k,v in b.items()}
                logits.append(model(b['input_values'],
                                     attention_mask=b.get('attention_mask')).logits.cpu().numpy())
        mt=max(l.shape[1] for l in logits)
        lg=np.concatenate([np.pad(l,((0,0),(0,mt-l.shape[1]),(0,0))) for l in logits],0)
        preds=greedy_ctc(lg, self.id2phone, self.phone2id, penalty=self.penalty)
        m=_full_metrics_inline(self.valid_c, self.valid_t, preds)
        m['epoch']=float(state.epoch)
        # Subgroup PER (no-error = FP source, error = TP source)
        if self.error_mask is not None:
            ei=[i for i,x in enumerate(self.error_mask) if x]
            ni=[i for i,x in enumerate(self.error_mask) if not x]
            m['per_err']  =evaluate_on_valid([self.valid_c[i] for i in ei],
                                             [self.valid_t[i] for i in ei],
                                             [preds[i] for i in ei])['per']
            m['per_noerr']=evaluate_on_valid([self.valid_c[i] for i in ni],
                                             [self.valid_t[i] for i in ni],
                                             [preds[i] for i in ni])['per']
        # Log training loss for mismatch analysis
        train_loss_entries = [e['loss'] for e in state.log_history
                              if 'loss' in e and 'eval_loss' not in e]
        m['train_loss'] = float(train_loss_entries[-1]) if train_loss_entries else None
        self.history.append(m)
        print(f"  [epoch {state.epoch:.0f}] P={m['prec']:.4f} R={m['rec']:.4f} "
              f"F1={m['f1']:.4f} PER={m['per']:.4f} "
              f"PER_noerr={m.get('per_noerr',0):.4f} Score={m['score']:.4f}")
        # Best-checkpoint tracking
        if m['score'] > self._best_score:
            self._best_score = m['score']
            self._best_epoch = int(state.epoch)
            if self.best_ckpt_dir:
                Path(self.best_ckpt_dir).mkdir(parents=True, exist_ok=True)
                model.save_pretrained(self.best_ckpt_dir)
                if self._fe is not None:
                    self._fe.save_pretrained(self.best_ckpt_dir)
                print(f"  ↑ NEW BEST ep{self._best_epoch}: Score={m['score']:.4f}"
                      f" → {self.best_ckpt_dir}")
        if self.out_path:
            json.dump(self.history, open(self.out_path,'w'), indent=2)

def _get_s6_logits(valid_df, valid_ph, phone2id, p_trim):
    """Load Stage 6 validation logits: tries trainer first, then saved checkpoint."""
    _ds = MDDDataset(valid_df, valid_ph, phone2id, p_trim, lambda y: y, is_train=False)
    if 'trainer' in dir() and hasattr(trainer, 'model'):
        return trainer.predict(_ds).predictions
    for _ckpt in [STAGE6_BEST_CKPT,
                  '/kaggle/working/outputs/stage6_clean_eval/final_model']:
        if Path(_ckpt).exists():
            print(f'trainer not in scope — loading Stage 6 model from {_ckpt}')
            _fe  = AutoFeatureExtractor.from_pretrained(_ckpt)
            _m   = Wav2Vec2ForCTC.from_pretrained(_ckpt).to(device)
            _m.eval()
            _col = MDDCollator(fe=_fe)
            _ldr = torch.utils.data.DataLoader(_ds, batch_size=8,
                                                collate_fn=_col, shuffle=False)
            _ll  = []
            with torch.no_grad():
                for _b in _ldr:
                    _b = {k: v.to(device) if hasattr(v, 'to') else v for k, v in _b.items()}
                    _ll.append(_m(_b['input_values'],
                                  attention_mask=_b.get('attention_mask')).logits.cpu().numpy())
            _mt = max(l.shape[1] for l in _ll)
            _lg = np.concatenate([np.pad(l, ((0,0),(0,_mt-l.shape[1]),(0,0))) for l in _ll])
            del _m
            import gc; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            return _lg
    raise FileNotFoundError(
        'No Stage 6 checkpoint found. Run cell 25 (Stage 6 training) first.\n'
        f'  Tried: {STAGE6_BEST_CKPT}\n'
        '  Tried: /kaggle/working/outputs/stage6_clean_eval/final_model')

print('EpochMetricsCallback + _get_s6_logits defined.')



## 7. Execute Selected Stage

In [ ]:
# ── STAGE 1: Minimal Baseline ─────────────────────────────────────────────────
if RUN_STAGE == 'stage1':
    preproc, augment, use_adult = build_audio_pipeline('stage1')
    t_df = train_df; t_ph = train_ph

    result_s1 = run_experiment(
        exp_name    = 'stage1_baseline',
        model_key   = 'wav2vec2-vn',
        train_df    = t_df, train_ph = t_ph,
        valid_df    = valid_df, valid_ph = valid_ph,
        preproc_fn  = preproc,
        augment_fn  = augment,
    )

    # ── Stage 1 exit check ───────────────────────────────────────────────────
    s = result_s1['score']
    B0 = 0.4994
    print(f'\nStage 1 Score: {s:.4f}  (B0={B0})')
    if s > B0:
        print('✅ Beat B0 — proceed to Stage 2')
    elif s > 0.45:
        print('⚠️  Below B0 but above 0.45 — check blank ratio, try head_lr=5e-3')
    else:
        print('❌ Below 0.45 — debug: check label encoding, CTC loss, vocab')

In [ ]:
# ── STAGE 2: Full Ablation Loop ──────────────────────────────────────────────
# Runs ALL ablations (A–F) sequentially, each INDEPENDENT from Stage 1 baseline.
# Each experiment: fresh model from pretrained, same 10 epochs, same valid set.
if RUN_STAGE == 'stage2':
    import gc

    ABLATIONS_TO_RUN = ['A', 'B', 'C', 'D', 'E', 'F']
    ablation_results = {}

    # Load Stage 1 baseline score for delta comparison
    _log = json.load(open('/kaggle/working/results/results_log.json')) \
           if Path('/kaggle/working/results/results_log.json').exists() else {}
    S1_SCORE = _log.get('stage1_baseline', {}).get('score', 0.4994)
    print(f'Stage 1 baseline score: {S1_SCORE:.4f}')
    print(f'Running ablations: {ABLATIONS_TO_RUN}')
    print('='*60)

    for abl in ABLATIONS_TO_RUN:
        print(f'\n>>> Ablation {abl} starting...')
        enable_spec = (abl == 'E')
        preproc, augment, use_adult = build_audio_pipeline('stage2', ablation=abl)

        # Each ablation uses its own data slice
        t_df = train_df[train_df['speaker_id'] != 'ADULT'].reset_index(drop=True) \
               if not use_adult else train_df
        t_ph = train_ph[train_ph['speaker_id'] != 'ADULT'].reset_index(drop=True) \
               if not use_adult else train_ph

        try:
            result = run_experiment(
                exp_name          = f'stage2_ablation_{abl}',
                model_key         = 'wav2vec2-vn',
                train_df          = t_df, train_ph = t_ph,
                valid_df          = valid_df, valid_ph = valid_ph,
                preproc_fn        = preproc,
                augment_fn        = augment,
                enable_spec_augment = enable_spec,
            )
            ablation_results[abl] = result
        except Exception as e:
            print(f'Ablation {abl} FAILED: {e}')
            ablation_results[abl] = {'score': float('nan'), 'error': str(e)}

        # Free GPU memory between experiments
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ── Summary table ────────────────────────────────────────────────────────
    print('\n' + '='*65)
    print(f'STAGE 2 ABLATION SUMMARY  (S1 baseline={S1_SCORE:.4f}, B0=0.4994)')
    print('='*65)
    print(f'{"Ablation":12s}  {"F1":7s}  {"PER":7s}  {"DER":7s}  {"Score":7s}  {"Delta":7s}  Keep?')
    print('-'*65)
    for abl, res in ablation_results.items():
        if 'error' in res:
            print(f'{"  " + abl:12s}  ERROR: {res["error"][:40]}')
            continue
        delta = res['score'] - S1_SCORE
        keep  = '✅ KEEP' if delta >= 0.010 else ('⚠️  borderline' if delta >= 0 else '❌ skip')
        print(f'  {abl:10s}  {res["f1"]:.4f}  {res["per"]:.4f}  {res["der"]:.4f}  '
              f'{res["score"]:.4f}  {delta:+.4f}  {keep}')
    print('='*65)
    print('Decision rule: Keep if Delta >= +0.010')


In [ ]:
# ── STAGE 3: Full Model Comparison Loop ─────────────────────────────────────
# Runs all 3 models with best_preproc = B+F (trim_silence + no_adult).
# wav2vec2-vn reference: Stage 1 Score=0.5045 (no preprocessing).
if RUN_STAGE == 'stage3':
    import gc

    STAGE3_MODELS = ['wav2vec2-vn', 'hubert', 'wav2vec2-100h']
    stage3_results = {}

    # Load previous results for reference
    _rpath = Path('/kaggle/working/results/results_log.json')
    _log   = json.load(open(_rpath)) if _rpath.exists() else {}
    S1_REF = _log.get('stage1_baseline', {}).get('score', 0.5045)
    print(f'Stage 1 reference score: {S1_REF:.4f}  (B0=0.4994)')
    print(f'best_preproc: B (trim_silence) + F (no_adult)')
    print(f'Running models: {STAGE3_MODELS}')
    print('='*60)

    for model_key in STAGE3_MODELS:
        print(f'\n>>> Model {model_key} starting...')
        preproc, augment, use_adult = build_audio_pipeline('stage3')
        # use_adult=False from stage3 pipeline (excludes ADULT group)
        t_df = train_df[train_df['speaker_id'] != 'ADULT'].reset_index(drop=True)
        t_ph = train_ph[train_ph['speaker_id'] != 'ADULT'].reset_index(drop=True)

        try:
            result = run_experiment(
                exp_name  = f'stage3_{model_key}',
                model_key = model_key,
                train_df  = t_df, train_ph = t_ph,
                valid_df  = valid_df, valid_ph = valid_ph,
                preproc_fn = preproc,
                augment_fn = augment,
            )
            stage3_results[model_key] = result
        except Exception as e:
            print(f'Model {model_key} FAILED: {e}')
            stage3_results[model_key] = {'score': float('nan'), 'error': str(e)}

        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    # ── Comparison table ──────────────────────────────────────────────────────
    print('\n' + '='*70)
    print(f'STAGE 3 MODEL COMPARISON  (best_preproc=B+F, B0=0.4994)')
    print('='*70)
    print(f'{"Model":22s}  {"F1":7s}  {"PER":7s}  {"DER":7s}  {"Score":7s}  {"Δ vs B0":8s}')
    print('-'*70)
    # Reference rows
    print(f'{"B0 canonical":22s}  {0.0:.4f}  {0.006:.4f}  {0.0:.4f}  {0.4994:.4f}  {0.0:+.4f}')
    print(f'{"S1 wav2vec2-vn (no prep)":22s}  {0.0745:.4f}  {0.1278:.4f}  {0.050:.4f}  {0.5045:.4f}  {+0.0051:+.4f}')
    for mk, res in stage3_results.items():
        if 'error' in res:
            print(f'{mk:22s}  ERROR: {res["error"][:40]}')
            continue
        delta = res['score'] - 0.4994
        print(f'{mk:22s}  {res["f1"]:.4f}  {res["per"]:.4f}  {res["der"]:.4f}  '
              f'{res["score"]:.4f}  {delta:+.4f}')
    print('='*70)


In [ ]:
# ── STAGE 4v2: Critical Re-examination ──────────────────────────────────────
# Each test challenges a specific causal claim from previous analysis.
# No model retraining needed — uses saved predictions.

if RUN_STAGE == 'stage4_diag':
    import gc, sys, csv, tempfile
    sys.path.insert(0, '/kaggle/working/mdd-challenge')
    import evaluate as ev

    # ── Load model + generate predictions (same as before) ──────────────────
    for model_path in [
        '/kaggle/working/outputs/stage2_ablation_B/final_model',
        '/kaggle/working/mdd-challenge/outputs/stage2_ablation_B/final_model',
        '/kaggle/working/outputs/stage3_wav2vec2-vn/final_model',
        '/kaggle/working/mdd-challenge/outputs/stage3_wav2vec2-vn/final_model',
    ]:
        if Path(model_path).exists():
            diag_fe    = AutoFeatureExtractor.from_pretrained(model_path)
            diag_model = Wav2Vec2ForCTC.from_pretrained(model_path).to(device)
            diag_model.eval(); print(f'Loaded: {model_path}'); break
    else: raise FileNotFoundError('No saved model found')

    def p_trim(y): return trim_silence(y, 16000)[:MAX_SAMPLES]
    diag_ds  = MDDDataset(valid_df, valid_ph, phone2id, p_trim, lambda y:y, is_train=False)
    loader   = torch.utils.data.DataLoader(
        diag_ds, batch_size=8, collate_fn=MDDCollator(fe=diag_fe), shuffle=False)
    all_logits = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) if hasattr(v,'to') else v for k,v in batch.items()}
            out = diag_model(input_values=batch['input_values'],
                             attention_mask=batch.get('attention_mask'))
            all_logits.append(out.logits.cpu().numpy())
    max_t = max(l.shape[1] for l in all_logits)
    logits_all = np.concatenate(
        [np.pad(l,((0,0),(0,max_t-l.shape[1]),(0,0))) for l in all_logits], axis=0)
    predictions = greedy_ctc(logits_all, id2phone, phone2id, penalty=0.0)

    # Helper: write CSVs and run evaluate.py returning P, R, F1
    def full_metrics(gt_c, gt_t, preds, label=''):
        gp = '/tmp/_gt2.csv'; rp = '/tmp/_res2.csv'
        with open(gp,'w',newline='') as f:
            w=csv.DictWriter(f,['canonical','transcript']); w.writeheader()
            for c,t in zip(gt_c,gt_t): w.writerow({'canonical':c,'transcript':t})
        with open(rp,'w',newline='') as f:
            w=csv.DictWriter(f,['predict']); w.writeheader()
            for p in preds: w.writerow({'predict':p})
        # Inline precision/recall — ev.compute_f1 returns only float
        gt_rows  = list(csv.DictReader(open(gp)))
        res_rows = list(csv.DictReader(open(rp)))
        cor_nocor=sub_sub=sub_sub1=sub_nosub=0
        ins_ins=ins_ins1=ins_noins=0
        del_del=del_del1=del_nodel=0
        for gr,rr in zip(gt_rows,res_rows):
            ref_s,hum_s,our_s = gr['canonical'],gr['transcript'],rr['predict']
            from evaluate import _align_pair
            rs,hs,op_rh = _align_pair(ref_s,hum_s)
            hs2,os2,op_ho= _align_pair(hum_s,our_s)
            rs3,os3,op_ro= _align_pair(ref_s,our_s)
            flag=0
            for ii in range(len(rs)):
                if rs[ii]=='<eps>': continue
                while flag<len(rs3) and rs3[flag]=='<eps>': flag+=1
                if flag<len(rs3) and rs[ii]==rs3[flag]:
                    if op_rh[ii]=='D' and op_ro[flag]=='D': del_del+=1
                    elif op_rh[ii]=='D' and op_ro[flag]!='D' and op_ro[flag]!='C': del_del1+=1
                    elif op_rh[ii]=='D' and op_ro[flag]=='C': del_nodel+=1
                    flag+=1
            flag=0
            for ii in range(len(hs)):
                if hs[ii]=='<eps>': continue
                while flag<len(hs2) and hs2[flag]=='<eps>': flag+=1
                if flag<len(hs2) and hs[ii]==hs2[flag]:
                    if op_rh[ii]=='C' and op_ho[flag]!='C': cor_nocor+=1
                    if op_rh[ii]=='S' and op_ho[flag]=='C': sub_sub+=1
                    elif op_rh[ii]=='S' and op_ho[flag]!='C' and rs[ii]!=os2[flag]: sub_sub1+=1
                    elif op_rh[ii]=='S' and op_ho[flag]!='C' and rs[ii]==os2[flag]: sub_nosub+=1
                    if op_rh[ii]=='I' and op_ho[flag]=='C': ins_ins+=1
                    elif op_rh[ii]=='I' and op_ho[flag]!='C' and op_ho[flag]!='D': ins_ins1+=1
                    elif op_rh[ii]=='I' and op_ho[flag]=='D': ins_noins+=1
                    flag+=1
        TR=sub_sub+sub_sub1+del_del+del_del1+ins_ins+ins_ins1
        FR=cor_nocor; FA=sub_nosub+ins_noins+del_nodel
        prec = TR/(TR+FR) if (TR+FR)>0 else 0.0
        rec  = TR/(TR+FA) if (TR+FA)>0 else 0.0
        f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
        per = ev.compute_per(gp,rp)
        der = ev.compute_der(gp,rp)
        sc  = 0.5*f1+0.4*(1-der)+0.1*(1-per)
        if label: print(f'{label:40s}  P={prec:.4f}  R={rec:.4f}  F1={f1:.4f}  '
                        f'PER={per:.4f}  Score={sc:.4f}')
        return {'f1':f1,'prec':prec,'rec':rec,'per':per,'der':der,'score':sc}


    # ── CHALLENGE 1: Measure actual Precision/Recall ─────────────────────────
    print('='*65)
    print('CHALLENGE 1: Actual Precision and Recall (never measured before)')
    print('='*65)
    error_mask = valid_ph['ph_error'].values.astype(bool)
    err_idx    = [i for i,m in enumerate(error_mask) if m]
    noerr_idx  = [i for i,m in enumerate(error_mask) if not m]

    full_metrics(VALID_C, VALID_T, predictions, 'Global (all 460 samples)')
    full_metrics([VALID_C[i] for i in err_idx], [VALID_T[i] for i in err_idx],
                 [predictions[i] for i in err_idx], 'Error samples only (n=46)')
    full_metrics([VALID_C[i] for i in noerr_idx], [VALID_T[i] for i in noerr_idx],
                 [predictions[i] for i in noerr_idx], 'No-error samples only (n=414)')

    # ── CHALLENGE 2: Direct FP measurement on no-error samples ───────────────
    print()
    print('='*65)
    print('CHALLENGE 2: Direct false positive count on no-error samples')
    print('(Not estimated from PER — measured from evaluate.py alignment)')
    print('='*65)
    # For no-error samples: canonical=transcript, any detected error = false positive
    # Run per-sample and count how many have F1>0 (= false alarm)
    fp_count = 0; fp_positions_total = 0
    noerr_per_list = []
    for idx in noerr_idx:
        c = VALID_C[idx]; t = VALID_T[idx]; p = predictions[idx]
        m = full_metrics([c],[t],[p])
        noerr_per_list.append(m['per'])
        # On no-error samples: transcript=canonical, so F1>0 means false detection
        # Count divergences from canonical directly
        c_toks = c.split(); p_toks = p.split() if p else []
        # Simple edit distance count (NW-free approximation for speed)
        diff = sum(1 for ct,pt in zip(c_toks,p_toks) if ct!=pt) + abs(len(c_toks)-len(p_toks))
        if diff > 0: fp_count += 1
        fp_positions_total += diff
    print(f'No-error samples with ≥1 divergence from canonical: {fp_count}/{len(noerr_idx)}')
    print(f'Total divergence positions on no-error samples: {fp_positions_total}')
    print(f'Average divergences/sample: {fp_positions_total/len(noerr_idx):.2f}')
    print(f'Average PER on no-error samples: {np.mean(noerr_per_list):.4f}')
    print()
    print('If FP >> TP (34): confirms precision bottleneck hypothesis')
    print('If FP ≈ TP: precision not the main issue')

    # ── CHALLENGE 3: Threshold sweep — TP suppression vs FP suppression ──────
    print()
    print('='*65)
    print('CHALLENGE 3: Threshold sweep — measures TP vs FP suppression')
    print('If threshold suppresses more TPs than FPs → NOT a valid fix')
    print('='*65)

    def edit_dist_fast(s1, s2):
        t1,t2=s1.split(),s2.split()
        if t1==t2: return 0
        n,m=len(t1),len(t2)
        dp=list(range(m+1))
        for i in range(1,n+1):
            prev,dp[0]=dp[0],i
            for j in range(1,m+1):
                prev,dp[j]=dp[j],prev if t1[i-1]==t2[j-1] else 1+min(prev,dp[j],dp[j-1])
        return dp[m]

    print(f'{"Thresh":7s}  {"F1":7s}  {"Prec":7s}  {"Rec":7s}  {"Score":7s}  '
          f'{"TP_kept%":9s}  {"FP_suppressed%":14s}')
    print('-'*75)

    # Baseline: no threshold
    base = full_metrics(VALID_C, VALID_T, predictions)

    # Count how many error samples the model currently "detects" (pred≠canonical)
    curr_err_detections   = sum(1 for i in err_idx   if edit_dist_fast(predictions[i], VALID_C[i]) > 0)
    curr_noerr_detections = sum(1 for i in noerr_idx if edit_dist_fast(predictions[i], VALID_C[i]) > 0)

    for k in [0, 1, 2, 3, 4, 5]:
        # Apply threshold: if dist(pred,canonical) <= k, output canonical
        new_preds = [
            predictions[i] if edit_dist_fast(predictions[i], VALID_C[i]) > k else VALID_C[i]
            for i in range(len(predictions))
        ]
        m = full_metrics(VALID_C, VALID_T, new_preds)

        # Measure TP suppression: error samples where pred was changed to canonical
        tp_suppressed = sum(
            1 for i in err_idx
            if edit_dist_fast(predictions[i], VALID_C[i]) <= k
               and edit_dist_fast(predictions[i], VALID_C[i]) > 0  # was detecting before
        )
        fp_suppressed = sum(
            1 for i in noerr_idx
            if edit_dist_fast(predictions[i], VALID_C[i]) <= k
               and edit_dist_fast(predictions[i], VALID_C[i]) > 0  # was detecting before
        )
        tp_pct = tp_suppressed/max(1,curr_err_detections)*100
        fp_pct = fp_suppressed/max(1,curr_noerr_detections)*100
        print(f'k={k:<5d}  {m["f1"]:.4f}  {m["prec"]:.4f}  {m["rec"]:.4f}  {m["score"]:.4f}  '
              f'{100-tp_pct:8.1f}%   {fp_pct:12.1f}%')

    print('TP_kept% = % of error-sample detections preserved')
    print('FP_suppressed% = % of no-error-sample detections removed')
    print('Good threshold: FP_suppressed >> TP_suppressed')

    # ── CHALLENGE 4: PER-F1 correlation — sample difficulty confound ─────────
    print()
    print('='*65)
    print('CHALLENGE 4: PER-F1 — sample difficulty confound test')
    print('If easy samples drive both low-PER and high-F1, correlation is spurious')
    print('='*65)
    # For error samples: compare PER to transcript vs PER to canonical
    # Also check: sequence length (proxy for difficulty)
    per_f1_data = []
    for idx in err_idx:
        c=VALID_C[idx]; t=VALID_T[idx]; p=predictions[idx]
        if not p: continue
        per_to_t = edit_dist_fast(p,t) / max(1, len(t.split()))
        per_to_c = edit_dist_fast(p,c) / max(1, len(c.split()))
        m = full_metrics([c],[t],[p])
        seq_len = len(c.split())
        n_gt_errors = edit_dist_fast(c,t)
        per_f1_data.append({'per':per_to_t,'f1':m['f1'],
                             'seq_len':seq_len,'n_gt_errors':n_gt_errors,
                             'per_to_canon':per_to_c})

    import pandas as pd
    pf = pd.DataFrame(per_f1_data)
    print('Correlation matrix (error samples only):')
    print(pf[['per','f1','seq_len','n_gt_errors']].corr().round(3))
    print()
    print('If corr(seq_len, per) > 0.5 AND corr(seq_len, f1) < 0 → sample difficulty confound')
    print('If corr(per, f1) < -0.3 AND corr(seq_len, per) < 0.3 → causal PER→F1 more likely')

    # ── CHALLENGE 5: Canonical bias at MISSED positions ───────────────────────
    print()
    print('='*65)
    print('CHALLENGE 5: What does model predict at MISSED error positions?')
    print('Missed = GT error positions where model did NOT diverge from canonical')
    print('='*65)

    def _align_nw(seq1, seq2):
        GAP,MATCH,MM=-1,1,-1
        n,m=len(seq1),len(seq2)
        sc=[[0]*(n+1) for _ in range(m+1)]
        for i in range(m+1): sc[i][0]=GAP*i
        for j in range(n+1): sc[0][j]=GAP*j
        for i in range(1,m+1):
            for j in range(1,n+1):
                s=MATCH if seq1[j-1]==seq2[i-1] else(GAP if '<eps>'in(seq1[j-1],seq2[i-1]) else MM)
                sc[i][j]=max(sc[i-1][j-1]+s,sc[i-1][j]+GAP,sc[i][j-1]+GAP)
        a1,a2=[],[]; i,j=m,n
        while i>0 and j>0:
            s=MATCH if seq1[j-1]==seq2[i-1] else(GAP if '<eps>'in(seq1[j-1],seq2[i-1]) else MM)
            if sc[i][j]==sc[i-1][j-1]+s: a1.append(seq1[j-1]);a2.append(seq2[i-1]);i-=1;j-=1
            elif sc[i][j]==sc[i][j-1]+GAP: a1.append(seq1[j-1]);a2.append('<eps>');j-=1
            else: a1.append('<eps>');a2.append(seq2[i-1]);i-=1
        while j>0: a1.append(seq1[j-1]);a2.append('<eps>');j-=1
        while i>0: a1.append('<eps>');a2.append(seq2[i-1]);i-=1
        return a1[::-1],a2[::-1]

    missed_canonical  = 0  # model predicted canonical phoneme at missed position
    missed_other      = 0  # model predicted something else (neither canonical nor transcript)
    missed_total      = 0

    for idx in err_idx:
        c=VALID_C[idx]; t=VALID_T[idx]; p=predictions[idx]
        c_toks=c.split(); t_toks=t.split(); p_toks=p.split() if p else []

        # GT errors: positions where canonical != transcript (substitution only)
        ac,at = _align_nw(c_toks,t_toks)
        ac2,ap = _align_nw(c_toks,p_toks)

        for k,(rc,rt) in enumerate(zip(ac,at)):
            if rc==rt or rc=='<eps>' or rt=='<eps>': continue  # not a substitution
            missed_total += 1
            # What did model predict at this aligned position?
            if k < len(ap):
                if ac2[k] == rc and ap[k] == rc:  # model matches canonical = missed detection
                    missed_canonical += 1
                elif ap[k] != rt:  # model predicted neither canonical nor transcript
                    missed_other += 1
                # else: model predicted transcript phoneme but alignment didn't credit it

    print(f'Total GT substitution positions: {missed_total}')
    print(f'Model predicted canonical (missed): {missed_canonical} ({missed_canonical/max(1,missed_total)*100:.1f}%)')
    print(f'Model predicted other (not canon, not transcript): {missed_other} ({missed_other/max(1,missed_total)*100:.1f}%)')
    print(f'Implied predicted transcript correctly: '
          f'{missed_total-missed_canonical-missed_other} '
          f'({(missed_total-missed_canonical-missed_other)/max(1,missed_total)*100:.1f}%)')
    print()
    print('If missed_canonical > 50% → canonical bias IS relevant at missed positions')
    print('If missed_other > 50% → random prediction errors, not canonical bias')

    del diag_model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
# ── STAGE 5: Extended training (30 epochs) + per-epoch P/R/F1 curve ──────────
# Best config from Stage 1-3: wav2vec2-vn + trim_silence (Stage2-B, Score=0.5127).
# Stage2-B INCLUDED adult (ablation B → use_adult=True), so we keep full train_df.
# Decisive test: does Precision climb with epochs or plateau (loss-metric mismatch)?
if RUN_STAGE == 'stage5':
    import gc
    EPOCH_LOG = '/kaggle/working/results/stage5_epoch_metrics.json'
    err_mask = valid_ph['ph_error'].values.astype(bool)
    cb = EpochMetricsCallback(VALID_C, VALID_T, id2phone, phone2id,
                              penalty=0.0, out_path=EPOCH_LOG, error_mask=err_mask)

    def p_trim(y): return trim_silence(y, 16000)[:MAX_SAMPLES]
    def noop(y):   return y

    result_s5 = run_experiment(
        exp_name   = 'stage5_extended_30ep',
        model_key  = 'wav2vec2-vn',
        train_df   = train_df, train_ph = train_ph,   # full train incl. adult
        valid_df   = valid_df, valid_ph = valid_ph,
        preproc_fn = p_trim, augment_fn = noop,
        epochs     = 30,
        epoch_callback = cb,
    )

    # ── Per-epoch curve table ─────────────────────────────────────────────────
    print('\n' + '='*78)
    print('STAGE 5 PER-EPOCH CURVE  (best=wav2vec2-vn + trim_silence, 30 epochs)')
    print('='*78)
    print(f'{"Ep":4s} {"P":7s} {"R":7s} {"F1":7s} {"PER":7s} '
          f'{"PER_noerr":9s} {"PER_err":8s} {"Score":7s}')
    print('-'*78)
    for h in cb.history:
        print(f"{h['epoch']:<4.0f} {h['prec']:.4f}  {h['rec']:.4f}  {h['f1']:.4f}  "
              f"{h['per']:.4f}  {h.get('per_noerr',0):.4f}    "
              f"{h.get('per_err',0):.4f}   {h['score']:.4f}")
    print('='*78)
    # ── Auto-diagnosis ────────────────────────────────────────────────────────
    if len(cb.history) >= 5:
        precs = [h['prec'] for h in cb.history]
        pn    = [h.get('per_noerr',0) for h in cb.history]
        last5_p = precs[-5:]; first_half = precs[:len(precs)//2]; second_half = precs[len(precs)//2:]
        p_still_rising = (np.mean(second_half) - np.mean(first_half)) > 0.01
        pn_still_dropping = (pn[len(pn)//2] - pn[-1]) > 0.01 if len(pn) > 2 else False
        print('DIAGNOSIS:')
        if p_still_rising:
            print('  ✅ Precision still climbing → MORE TRAINING HELPS. Extend epochs.')
        elif pn_still_dropping:
            print('  ⚠️  Precision plateaued but PER_noerr still dropping →')
            print('     LOSS-METRIC MISMATCH. Need weighted CTC / confidence decoding (Stage 6).')
        else:
            print('  ❌ Both plateaued → training alone insufficient. Rethink approach (Stage 6).')
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
# ── STAGE 6: Clean eval protocol — 30 epochs, best-ckpt, fixed penalty ─────
if RUN_STAGE == 'stage6':
    import gc
    err_mask = valid_ph['ph_error'].values.astype(bool)
    cb6 = EpochMetricsCallback(
        VALID_C, VALID_T, id2phone, phone2id,
        penalty=FIXED_PENALTY,
        out_path=EPOCH_LOG_S6,
        error_mask=err_mask,
        best_ckpt_dir=STAGE6_BEST_CKPT,
    )

    def p_trim(y): return trim_silence(y, 16000)[:MAX_SAMPLES]

    result_s6 = run_experiment(
        exp_name       = 'stage6_clean_eval',
        model_key      = 'wav2vec2-vn',
        train_df       = train_df, train_ph = train_ph,
        valid_df       = valid_df, valid_ph = valid_ph,
        preproc_fn     = p_trim, augment_fn = lambda y: y,
        epochs         = 30,
        epoch_callback = cb6,
    )

    s5_score = 0.5217  # Stage 5 final (penalty-tuned, for reference)
    print(f'\nStage 5 (penalty-tuned):  Score = {s5_score:.4f}')
    print(f'Stage 6 (fixed penalty):  Score = {result_s6["score"]:.4f}')
    print(f'Stage 6 best epoch:       ep{cb6._best_epoch} (Score={cb6._best_score:.4f})')

    # ── Per-epoch table ───────────────────────────────────────────────────────
    print('\n' + '='*78)
    print('STAGE 6 PER-EPOCH CURVE  (fixed penalty, best-ckpt tracking)')
    print('='*78)
    print(f'{"Ep":4s} {"P":7s} {"R":7s} {"F1":7s} {"PER":7s} '
          f'{"PER_noerr":9s} {"PER_err":8s} {"Score":7s} {"train_loss":10s}')
    print('-'*90)
    for h in cb6.history:
        tl = f"{h.get('train_loss',0):.4f}" if h.get('train_loss') else 'N/A'
        print(f"{h['epoch']:<4.0f} {h['prec']:.4f}  {h['rec']:.4f}  {h['f1']:.4f}  "
              f"{h['per']:.4f}  {h.get('per_noerr',0):.4f}    "
              f"{h.get('per_err',0):.4f}   {h['score']:.4f}  {tl}")
    print('='*78)

    # ── Corrected auto-diagnosis (slope test on last 10, not half-average) ────
    if len(cb6.history) >= 10:
        import numpy as np
        precs = [h['prec'] for h in cb6.history]
        last10 = precs[-10:]
        slope, _ = np.polyfit(range(10), last10, 1)
        resid = np.array(last10) - (np.polyfit(range(10), last10, 1)[1] + slope * np.arange(10))
        se = resid.std(ddof=2) / (np.arange(10) - 4.5).std()
        t_stat = slope / se if se > 0 else 0
        print(f'\nPrecision slope (last 10 epochs): {slope:.6f}/epoch  t={t_stat:.2f}')
        if t_stat > 2.0:
            print('DIAGNOSIS: ✅ Precision trend significant (t>2) — training still helps.')
        else:
            print('DIAGNOSIS: ⚠️  Precision trend NOT significant (t≤2) — MISMATCH likely.')
            print('           Run Tasks 2-4 (FP audit, mismatch analysis, calibration).')
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
# ── STAGE 6 Task 2: FP Audit — decompose cor_nocor false positives ───────────
if RUN_STAGE == 'stage6':
    import sys
    sys.path.insert(0, '/kaggle/working/mdd-challenge')
    import importlib, sys
    if 'stage6_utils' in sys.modules:
        importlib.reload(sys.modules['stage6_utils'])
    from stage6_utils import audit_fp_sources, fp_summary

    # Use Stage 6 logits (load from trainer if available, else from checkpoint)
    if 's6_logits' not in dir():
        s6_logits = _get_s6_logits(valid_df, valid_ph, phone2id, p_trim)

    s6_preds = greedy_ctc(s6_logits, id2phone, phone2id, penalty=FIXED_PENALTY)

    err_idx  = [i for i, x in enumerate(err_mask) if x]
    nerr_idx = [i for i, x in enumerate(err_mask) if not x]

    print('='*60)
    print('FP AUDIT — Stage 6 predictions')
    print('='*60)
    audit_all = audit_fp_sources(VALID_C, VALID_T, s6_preds)
    fp_summary(audit_all)

    print(f'\n── FP from NON-ERROR utterances ({len(nerr_idx)} samples) ──')
    audit_nerr = audit_fp_sources(
        [VALID_C[i] for i in nerr_idx], [VALID_T[i] for i in nerr_idx],
        [s6_preds[i] for i in nerr_idx])
    fp_summary(audit_nerr)

    print(f'\n── FP from ERROR utterances ({len(err_idx)} samples) ──')
    audit_err = audit_fp_sources(
        [VALID_C[i] for i in err_idx], [VALID_T[i] for i in err_idx],
        [s6_preds[i] for i in err_idx])
    fp_summary(audit_err)

    fp_nerr_total = audit_nerr['total_fp']
    fp_err_total  = audit_err['total_fp']
    fp_total      = audit_all['total_fp']
    print(f'\n── FP Source Attribution ──')
    print(f'  Non-error utterances: {fp_nerr_total}/{fp_total} = {fp_nerr_total/max(1,fp_total)*100:.1f}%')
    print(f'  Error utterances:     {fp_err_total}/{fp_total} = {fp_err_total/max(1,fp_total)*100:.1f}%')
    print('\nDIAGNOSIS:')
    if fp_nerr_total / max(1, fp_total) > 0.70:
        print('  ✅ FP dominated by non-error utterances → ASR errors on clean speech')
        print('     Precision bottleneck = ASR accuracy, not MDD logic')
    else:
        print('  ⚠️  Significant FP from error utterances — check alignment artifacts')


In [ ]:
# ── STAGE 6 Task 3: Loss–Metric Mismatch Analysis ────────────────────────────
if RUN_STAGE == 'stage6':
    from stage6_utils import mismatch_analysis
    import os

    print('=== Stage 6 Mismatch Analysis ===')
    if os.path.exists(EPOCH_LOG_S6):
        mismatch_analysis(EPOCH_LOG_S6, start_epoch=10)
    else:
        print('Stage 6 epoch log not found — run Stage 6 training first.')

    s5_json = '/kaggle/working/results/stage5_epoch_metrics.json'
    if os.path.exists(s5_json):
        print('\n=== Stage 5 Mismatch Analysis (reference, no train_loss) ===')
        mismatch_analysis(s5_json, start_epoch=10)


In [ ]:
# ── STAGE 6 Task 4: Calibration — Confidence Threshold P-R Sweep ─────────────
if RUN_STAGE == 'stage6':
    import importlib, sys
    if 'stage6_utils' in sys.modules:
        importlib.reload(sys.modules['stage6_utils'])
    from stage6_utils import pr_sweep

    if 's6_logits' not in dir():
        s6_logits = _get_s6_logits(valid_df, valid_ph, phone2id, p_trim)

    print('='*60)
    print('CALIBRATION: Confidence Threshold P-R Sweep (Stage 6 model)')
    print('='*60)
    sweep_results = pr_sweep(
        VALID_C, VALID_T, s6_logits, id2phone, phone2id,
        thresholds=[0.0, 0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
        blank_penalty=FIXED_PENALTY,
    )

    print('\n── Operating points with Precision ≥ 0.10 ──')
    viable = [(r['threshold'], r['precision'], r['recall'], r['f1'], r['score'])
              for r in sweep_results if r['precision'] >= 0.10]
    if viable:
        for thr, p, r, f1, sc in viable:
            print(f'  threshold={thr:.2f}  P={p:.4f}  R={r:.4f}  F1={f1:.4f}  Score={sc:.4f}')
        best_v = max(viable, key=lambda x: x[4])
        print(f'\nBest operating point: threshold={best_v[0]:.2f}, Score={best_v[4]:.4f}')
    else:
        print('  ❌ No threshold achieves Precision ≥ 0.10')
        print('     → Run Task 5 (GOP scoring) for deeper diagnosis.')

    max_p = max(r['precision'] for r in sweep_results)
    print(f'\nDIAGNOSIS:')
    if max_p >= 0.10:
        print(f'  ✅ Calibration achievable (max P={max_p:.4f}) — tune threshold for deployment.')
    else:
        print(f'  ❌ Max precision = {max_p:.4f} < 0.10 with any threshold.')
        print('     → FP source is structural. Proceed to Task 5 (GOP scoring).')


In [ ]:
# ── STAGE 6 Task 5 (Conditional): GOP Scoring ────────────────────────────────
# Run only if Task 4 cannot achieve Precision ≥ 0.10
if RUN_STAGE == 'stage6':
    import importlib, sys
    if 'stage6_utils' in sys.modules:
        importlib.reload(sys.modules['stage6_utils'])
    from stage6_utils import gop_pr_sweep

    if 's6_logits' not in dir():
        s6_logits = _get_s6_logits(valid_df, valid_ph, phone2id, p_trim)

    print('='*60)
    print('GOP SCORING: Log-likelihood ratio threshold sweep')
    print('='*60)
    gop_results = gop_pr_sweep(
        VALID_C, VALID_T, s6_logits, id2phone, phone2id,
        thresholds=[-5.0, -3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 5.0],
        blank_penalty=FIXED_PENALTY,
    )

    best_gop = max(gop_results, key=lambda x: x['score'])
    s6_ref_score = result_s6['score'] if 'result_s6' in dir() else 0.52
    print(f'\nBest GOP threshold: {best_gop["gop_threshold"]:.1f}, Score={best_gop["score"]:.4f}')
    print(f'Stage 6 base score:   {s6_ref_score:.4f}')
    print('\nDIAGNOSIS:')
    if best_gop['score'] > s6_ref_score + 0.005:
        print('  ✅ GOP scoring IMPROVES Score → confidence info separates errors.')
        print('     Next: implement GOP-based MDD module or GOP-weighted CTC loss.')
    else:
        print('  ❌ GOP scoring does not help → bottleneck is deeper than confidence.')
        print('     Next: detection head, cost-sensitive loss, or MDD-specific fine-tuning.')


In [ ]:
# ── STAGE 7A: FP Rate per Phoneme + Confusion Matrix ────────────────────────
if RUN_STAGE == 'stage6':
    import importlib, sys, numpy as np
    if 'stage6_utils' in sys.modules: importlib.reload(sys.modules['stage6_utils'])
    from stage6_utils import phoneme_fp_rate_table, phoneme_confusion_detail

    if 's6_logits' not in dir():
        s6_logits = _get_s6_logits(valid_df, valid_ph, phone2id, p_trim)
    s6_preds = greedy_ctc(s6_logits, id2phone, phone2id, penalty=FIXED_PENALTY)

    print('='*78)
    print('FP RATE PER PHONEME (sorted by FP_rate = FP_count / canonical_occurrences)')
    print('='*78)
    fp_cnt, occ, fp_rates, confusion = phoneme_fp_rate_table(
        VALID_C, VALID_T, s6_preds, top_n=20)

    all_rates  = list(fp_rates.values())
    median_rate = float(np.median(all_rates))
    mean_rate   = float(np.mean(all_rates))
    high_rate_phones = sorted(
        [ph for ph, r in fp_rates.items() if r > 2 * median_rate and fp_cnt[ph] >= 5],
        key=lambda x: -fp_rates[x]
    )
    print(f'\nMedian FP rate: {median_rate:.3f}  Mean: {mean_rate:.3f}')
    print(f'High-rate phonemes (rate > 2×median AND n≥5): {high_rate_phones[:15]}')

    # Group comparison: z-suffix vs others
    z_phones  = [p for p in fp_rates if p.endswith('z')]
    oth_phones = [p for p in fp_rates if not p.endswith('z')]
    z_fp   = sum(fp_cnt.get(p, 0) for p in z_phones)
    oth_fp = sum(fp_cnt.get(p, 0) for p in oth_phones)
    z_occ  = sum(occ.get(p, 0)   for p in z_phones)
    oth_occ= sum(occ.get(p, 0)   for p in oth_phones)
    total_fp = sum(fp_cnt.values())
    print(f'\n── FP contribution by phoneme group ──')
    print(f'  z-suffix : {z_fp}/{total_fp} FPs = {z_fp/total_fp*100:.1f}%'
          f'  occ={z_occ}  rate={z_fp/max(1,z_occ):.3f}')
    print(f'  other    : {oth_fp}/{total_fp} FPs = {oth_fp/total_fp*100:.1f}%'
          f'  occ={oth_occ}  rate={oth_fp/max(1,oth_occ):.3f}')

    # Confusion detail for high-rate phones
    print('\n── Confusion detail for high-rate phonemes ──')
    phoneme_confusion_detail(confusion, high_rate_phones[:15])

    print('\nDIAGNOSIS:')
    z_rate   = z_fp/max(1,z_occ)
    oth_rate = oth_fp/max(1,oth_occ)
    if z_rate > 2 * oth_rate:
        print('  ✅ z-suffix group FP rate significantly higher → anomalous error pattern')
        print('     Phoneme-group calibration with higher threshold for z-suffix is justified.')
    else:
        print('  ℹ️  z-suffix FP rate comparable to other groups → frequency confound')
        print(f'     Use high_rate_phones list for group calibration in Task 7B.')


In [ ]:
# ── STAGE 7B: Phoneme-Group Calibration ─────────────────────────────────────
if RUN_STAGE == 'stage6':
    import importlib, sys
    if 'stage6_utils' in sys.modules: importlib.reload(sys.modules['stage6_utils'])
    from stage6_utils import greedy_ctc_with_conf, group_calibration_sweep

    if 's6_logits' not in dir():
        s6_logits = _get_s6_logits(valid_df, valid_ph, phone2id, p_trim)
    base_preds, confidences = greedy_ctc_with_conf(
        s6_logits, id2phone, phone2id, blank_penalty=FIXED_PENALTY)

    # Use high_rate_phones from 7A; fallback to z-suffix if 7A not run
    Z_PHONES = ['nz','iz','ŋz','tz','ŋmz','mz','kz','uz','pz','z','ŋ̟z','kpz','k̟z']
    group = (high_rate_phones if 'high_rate_phones' in dir() and high_rate_phones
             else Z_PHONES)

    print('='*70)
    print(f'GROUP CALIBRATION: group={group[:6]}... ({len(group)} phonemes)')
    print('='*70)

    overall_best = {'score': -1}
    for default_thr in [0.70, 0.80, 0.90]:
        print(f'\n── default_threshold = {default_thr} ──')
        results = group_calibration_sweep(
            VALID_C, VALID_T, base_preds, confidences,
            group_phones=group,
            group_thresholds=[0.70, 0.80, 0.90, 0.95, 0.98, 1.0],
            default_threshold=default_thr)
        best = max(results, key=lambda x: x['score'])
        print(f'Best: grp_thr={best["grp_thr"]:.2f}, Score={best["score"]:.4f}, '
              f'P={best["precision"]:.4f}, R={best["recall"]:.4f}')
        if best['score'] > overall_best['score']:
            overall_best = {**best, 'default_thr': default_thr}

    print(f'\nOverall best: {overall_best}')
    print(f'Reference: Stage6-uniform-thr0.90=0.5443 | Stage5=0.5217 | Stage6-base=0.5100')

    print('\nDIAGNOSIS:')
    best_s = overall_best['score']
    if best_s > 0.5443:
        print(f'  ✅ Group calibration ({best_s:.4f}) BEATS uniform threshold (0.5443)')
        print('     Phoneme-specific approach is justified. Use this configuration.')
    elif best_s > 0.5118:
        print(f'  ℹ️  Group calibration ({best_s:.4f}) improves over base but not uniform thr.')
        print('     Recommend uniform threshold=0.90 (simpler, slightly better).')
    else:
        print(f'  ❌ Group calibration ({best_s:.4f}) ≤ base. Groups not useful.')
        print('     Proceed to targeted retraining guided by confusion matrix from 7A.')


In [ ]:
# ── STAGE 8A: Calibration Ceiling Sweep ─────────────────────────────────────
if RUN_STAGE == 'stage6':
    import importlib, sys
    if 'stage6_utils' in sys.modules: importlib.reload(sys.modules['stage6_utils'])
    from stage6_utils import calibration_ceiling_sweep, greedy_ctc_with_conf

    # Recompute fp_rates / fp_cnt if not already available from cell 30
    if 'fp_rates' not in dir() or 'fp_cnt' not in dir():
        from stage6_utils import phoneme_fp_rate_table
        if 's6_logits' not in dir():
            s6_logits = _get_s6_logits(valid_df, valid_ph, phone2id, p_trim)
        _s6p = greedy_ctc(s6_logits, id2phone, phone2id, penalty=FIXED_PENALTY)
        fp_cnt, occ, fp_rates, confusion = phoneme_fp_rate_table(VALID_C, VALID_T, _s6p, top_n=5)

    if 's6_logits' not in dir():
        s6_logits = _get_s6_logits(valid_df, valid_ph, phone2id, p_trim)
    base_preds, confidences = greedy_ctc_with_conf(
        s6_logits, id2phone, phone2id, blank_penalty=FIXED_PENALTY)

    print('='*70)
    print('CALIBRATION CEILING SWEEP (default_thr=0.90, grp_thr=1.0, vary K)')
    print('='*70)
    ceiling_sweep = calibration_ceiling_sweep(
        VALID_C, VALID_T, base_preds, confidences,
        fp_rates=fp_rates, fp_cnt=fp_cnt,
        default_threshold=0.90,
        k_values=[0, 5, 10, 15, 20, 25, 30, 35, 40, 50],
        min_fp_for_inclusion=5)

    best_k = max(ceiling_sweep, key=lambda x: x['score'])
    print(f'\nOptimal K={best_k["k"]}, Score={best_k["score"]:.4f}, '
          f'P={best_k["precision"]:.4f}, R={best_k["recall"]:.4f}')
    print(f'Reference: Stage7-K10=0.5477 | Stage6-uniform=0.5443 | Stage5=0.5217')

    ceiling = best_k['score']
    print(f'\nCALIBRATION CEILING ≈ {ceiling:.4f}')
    print('DIAGNOSIS:')
    if ceiling > 0.56:
        print('  ✅ Strong ceiling → calibration approach sufficient, deploy.')
    elif ceiling > 0.54:
        print('  ℹ️  Modest ceiling → calibration helpful but limited.')
        print('     Targeted retraining (tone-specific loss weighting) needed to go higher.')
    else:
        print('  ❌ Low ceiling → calibration exhausted. Must retrain.')


In [ ]:
# ── STAGE 8B: Tone-Group FP Analysis ────────────────────────────────────────
if RUN_STAGE == 'stage6':
    import importlib, sys
    if 'stage6_utils' in sys.modules: importlib.reload(sys.modules['stage6_utils'])
    from stage6_utils import tone_group_fp_analysis

    if 'fp_cnt' not in dir() or 'occ' not in dir():
        from stage6_utils import phoneme_fp_rate_table
        if 's6_logits' not in dir():
            s6_logits = _get_s6_logits(valid_df, valid_ph, phone2id, p_trim)
        _s6p = greedy_ctc(s6_logits, id2phone, phone2id, penalty=FIXED_PENALTY)
        fp_cnt, occ, fp_rates, confusion = phoneme_fp_rate_table(VALID_C, VALID_T, _s6p, top_n=5)

    print('='*72)
    print('TONE-GROUP FP ANALYSIS (group_rate = total_FP / total_occurrences)')
    print('='*72)
    group_rates = tone_group_fp_analysis(fp_cnt, occ, fp_rates)

    import numpy as np
    sorted_groups = sorted(group_rates.items(), key=lambda x: -x[1])
    print(f'\nRanking by group_rate:')
    for g, r in sorted_groups:
        bar = '█' * int(r * 60)
        print(f'  {g:12s}: {r:.3f}  {bar}')

    baseline_rate = group_rates.get('tone-0', 0.1) or 0.1
    high_tone_groups = [g for g, r in sorted_groups if r > 1.5 * baseline_rate]
    print(f'\nHigh-rate groups (rate > 1.5× tone-0={baseline_rate:.3f}): {high_tone_groups}')
    print('\nDIAGNOSIS:')
    if high_tone_groups:
        print(f'  → Suppress predictions for {high_tone_groups} groups as linguistic rule.')
        print('     Also consider: targeted retraining with tone-specific sampling.')
    else:
        print('  → No tone group is anomalously high. Use data-driven K from Task 8A.')


## 8. Results Summary

Chạy cell này sau khi tất cả experiments xong để xem bảng tổng hợp.

In [ ]:
if Path('results/results_log.json').exists():
    log = json.load(open('results/results_log.json'))
    B0 = log.get('naive_B0_canonical', {}).get('score', 0.4994)
    SEP = '='*70
    print(SEP)
    print(f"{'Experiment':35s}  {'F1':7s}  {'PER':7s}  {'DER':7s}  {'Score':7s}  {'dB0':7s}")
    print('-'*70)
    print(f"{'B0 canonical (trivial)':35s}  {0.0:.4f}  {0.0056:.4f}  {0.0:.4f}  {B0:.4f}  {0.0:+.4f}")
    skip = {'naive_B0_canonical','naive_B1_oracle','naive_B2_empty','split_info'}
    for k, v in log.items():
        if k in skip or not isinstance(v, dict) or 'score' not in v:
            continue
        delta = v['score'] - B0
        f1  = v.get('f1',  0)
        per = v.get('per', 0)
        der = v.get('der', 0)
        sc  = v['score']
        print(f"{k:35s}  {f1:.4f}  {per:.4f}  {der:.4f}  {sc:.4f}  {delta:+.4f}")
    print(SEP)
    print('Upper bound (oracle): Score=1.0000')
else:
    print('No results yet — run experiments first.')
